# Intelligent Patient Risk Assessment  
## Notebook 03 — Feature Engineering

This notebook creates clinically meaningful derived features to improve model performance and capture nonlinear health risk patterns.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

print("ibraries loaded")

ibraries loaded


In [2]:
from google.colab import files
import io

print("📁 Upload cleaned_data.csv")
uploaded = files.upload()

file_name = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[file_name]))

print("Dataset loaded")
df.head()

📁 Upload cleaned_data.csv


Saving cleaned_data.csv to cleaned_data.csv
Dataset loaded


,Age,gender,ethnicity,education_level,income_level,employment_status,smoking_status,alcohol_consumption_per_week,physical_activity_minutes_per_week,diet_score,...,triglycerides,glucose_fasting,glucose_postprandial,insulin_level,hba1c,diabetes_risk_score,hypertension_risk_score,heart_disease_risk_score,obesity_risk_score,cholesterol_risk_score
0,58,Male,Asian,Highschool,Lower-Middle,Employed,Never,0,215,5.7,...,145,136,236,6.36,8.18,51.716583,53.581369,50.796020,57.387241,51.492983
1,52,Female,White,Highschool,Middle,Employed,Former,1,143,6.7,...,30,93,150,2.00,5.63,26.096162,47.946740,23.853485,36.219202,3.984993
2,60,Male,Hispanic,Highschool,Middle,Unemployed,Never,1,57,6.4,...,36,118,195,5.07,7.51,54.830087,42.141104,38.977694,42.898797,21.793300
3,74,Female,Black,Highschool,Low,Retired,Never,0,49,3.4,...,140,139,253,5.28,9.03,60.729521,60.286946,53.579555,52.194082,22.962029
4,46,Male,White,Graduate,Middle,Retired,Never,1,109,7.2,...,160,137,184,12.74,7.20,32.778085,15.279074,18.721942,32.095465,34.296907


In [3]:
print("Shape:", df.shape)
print("Columns:", len(df.columns))

Shape: (97297, 33)
Columns: 33


In [4]:
risk_cols = [
    "diabetes_risk_score",
    "heart_disease_risk_score",
    "hypertension_risk_score",
    "obesity_risk_score",
    "cholesterol_risk_score"
]

existing_risks = [c for c in risk_cols if c in df.columns]

print("Detected risk columns:", existing_risks)

Detected risk columns: ['diabetes_risk_score', 'heart_disease_risk_score', 'hypertension_risk_score', 'obesity_risk_score', 'cholesterol_risk_score']


### Target Strategy

- Milestone 1 → diabetes_risk_score, heart_disease_risk_score  
- Milestone 2 → remaining risk scores  

Feature engineering below is designed to avoid target leakage and remain reusable.

In [5]:
age_cols = [c for c in df.columns if "age" in c.lower()]
age_col = age_cols[0] if age_cols else None
print("Age column:", age_col)

Age column: Age


In [6]:
if age_col:
    df["age_group"] = pd.cut(
        df[age_col],
        bins=[0, 30, 50, 120],
        labels=["young", "middle", "senior"]
    )
    print("Age group created")

Age group created


In [7]:
bmi_cols = [c for c in df.columns if "bmi" in c.lower()]
bmi_col = bmi_cols[0] if bmi_cols else None
print("BMI column:", bmi_col)

BMI column: bmi


In [8]:
def bmi_category(x):
    if x < 18.5:
        return "underweight"
    elif x < 25:
        return "normal"
    elif x < 30:
        return "overweight"
    else:
        return "obese"

if bmi_col:
    df["bmi_category"] = df[bmi_col].apply(bmi_category)
    print("BMI category created")

BMI category created


In [9]:
bp_cols = [c for c in df.columns if "bp" in c.lower()]

if bp_cols:
    bp_col = bp_cols[0]
    threshold = df[bp_col].quantile(0.75)
    df["high_bp_flag"] = (df[bp_col] > threshold).astype(int)
    print("BP risk flag created")
else:
    print("No BP column found")

BP risk flag created


In [10]:
df.shape

(97297, 36)

In [11]:
if age_col and bmi_col:
    df["bmi_age_interaction"] = df[age_col] * df[bmi_col]
    print("Interaction feature created")

Interaction feature created


In [12]:
available_risks = [c for c in risk_cols if c in df.columns]

if len(available_risks) >= 2:
    df["combined_risk_index"] = df[available_risks].mean(axis=1)
    print("Combined risk index created (use carefully during training)")
else:
    print("Not enough risk columns")

Combined risk index created (use carefully during training)


In [13]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
skewness = df[numeric_cols].skew().sort_values(ascending=False)
display(skewness.head(10))

,0
cardiovascular_history,3.119876
physical_activity_minutes_per_week,1.390451
family_history_diabetes,1.354286
high_bp_flag,1.157902
hypertension_history,1.149111
alcohol_consumption_per_week,0.702080
insulin_level,0.423257
bmi_age_interaction,0.408751
heart_disease_risk_score,0.363261
ldl_cholesterol,0.319729


In [14]:
new_cat_cols = [col for col in ["age_group", "bmi_category"] if col in df.columns]
print("New categorical columns:", new_cat_cols)

New categorical columns: ['age_group', 'bmi_category']


In [15]:
if new_cat_cols:
    df = pd.get_dummies(df, columns=new_cat_cols, drop_first=True)
    print("New categorical features encoded")

New categorical features encoded


In [16]:
print("Final shape:", df.shape)
df.head()

Final shape: (97297, 41)


,Age,gender,ethnicity,education_level,income_level,employment_status,smoking_status,alcohol_consumption_per_week,physical_activity_minutes_per_week,diet_score,...,obesity_risk_score,cholesterol_risk_score,high_bp_flag,bmi_age_interaction,combined_risk_index,age_group_middle,age_group_senior,bmi_category_obese,bmi_category_overweight,bmi_category_underweight
0,58,Male,Asian,Highschool,Lower-Middle,Employed,Never,0,215,5.7,...,57.387241,51.492983,1,1769.0,52.994839,False,True,True,False,False
1,52,Female,White,Highschool,Middle,Employed,Former,1,143,6.7,...,36.219202,3.984993,1,1201.2,27.620116,False,True,False,False,False
2,60,Male,Hispanic,Highschool,Middle,Unemployed,Never,1,57,6.4,...,42.898797,21.793300,0,1332.0,40.128196,False,True,False,False,False
3,74,Female,Black,Highschool,Low,Retired,Never,0,49,3.4,...,52.194082,22.962029,0,1983.2,49.950427,False,True,False,True,False
4,46,Male,White,Graduate,Middle,Retired,Never,1,109,7.2,...,32.095465,34.296907,0,975.2,26.634294,True,False,False,False,False


In [17]:
import os

os.makedirs("artifacts", exist_ok=True)

feature_path = "artifacts/feature_engineered_data.csv"
df.to_csv(feature_path, index=False)

print("Feature engineered dataset saved")

Feature engineered dataset saved


In [18]:
from google.colab import files
files.download("artifacts/feature_engineered_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Feature Engineering Completed

Features created are safe for multi-risk prediction and avoid target leakage.

These features will support:

- Milestone 1 → diabetes & heart risk models  
- Milestone 2 → additional risk models